# DermaVision AI: train on Google Colab

**Before you run anything:** Runtime > Change runtime type > **T4 GPU**. Then run the cells from top to bottom. The whole notebook takes roughly 30-60 minutes.

The notebook downloads the project code from GitHub, so there is nothing to upload. Training results are saved to your Google Drive in `My Drive/dermavision/models`, so they survive a disconnect.

In [ ]:
import torch

print("torch", torch.__version__)
assert torch.cuda.is_available(), (
    "No GPU found. Use Runtime > Change runtime type > T4 GPU, then run this cell again."
)
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
OUT_DIR = "/content/drive/MyDrive/dermavision/models"  # keep the folder name "models"

In [ ]:
REPO = "https://github.com/al-shovon/Derma-Vision-AI.git"  # change this if you fork the project

!rm -rf /content/dermavision-ai
!git clone -q {REPO} /content/dermavision-ai
%cd /content/dermavision-ai
!ls

## Get the HAM10000 dataset

The next cell downloads it with `kagglehub`. Public datasets should download without logging in.

If it asks for credentials: create an API token at https://www.kaggle.com/settings/api ("Generate New Token"), open the key icon (Secrets) in Colab's left sidebar, add a secret named `KAGGLE_API_TOKEN` with the token as its value, switch on Notebook access, and run the cell again.

No Kaggle at all? See the optional Dataverse cells after the next cell.

In [ ]:
!pip install -q -U kagglehub
import kagglehub

try:
    DATA = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")
except Exception as e:
    raise RuntimeError(
        "Download failed. Add a Kaggle API token as a Colab secret (see the text above) "
        "or use the Dataverse option below."
    ) from e
print("Dataset folder:", DATA)
!ls "{DATA}"

### Optional: download from the original source instead (skip if the cell above worked)

1. Open https://doi.org/10.7910/DVN/DBW86T (Harvard Dataverse).
2. For each of the three files `HAM10000_images_part_1` (zip), `HAM10000_images_part_2` (zip) and `HAM10000_metadata`, open its download menu and copy the download link (right-click, Copy link address).
3. Add a new code cell, paste your links over `PASTE_LINK_1/2/3`, and run it:

```
%%bash
mkdir -p /content/ham10000/images && cd /content/ham10000
wget -q -O part1.zip "PASTE_LINK_1"
wget -q -O part2.zip "PASTE_LINK_2"
wget -q -O HAM10000_metadata.csv "PASTE_LINK_3"
unzip -oq part1.zip -d images
unzip -oq part2.zip -d images
```

4. In another new code cell run `DATA = "/content/ham10000"`, then continue with the cells below.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

root = Path(DATA)
meta = next(root.rglob("HAM10000_metadata*"))
df = pd.read_csv(meta, sep=None, engine="python")
paths = {p.stem: p for p in root.rglob("*.jpg")}

print(len(df), "images,", df["lesion_id"].nunique(), "unique lesions")
print(df["dx"].value_counts())
print(df.groupby("lesion_id").size().value_counts().sort_index())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df["dx"].value_counts().plot.bar(ax=axes[0], title="Images per class")
df["age"].plot.hist(bins=20, ax=axes[1], title="Age")
df["localization"].value_counts().plot.barh(ax=axes[2], title="Body site")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 7, figsize=(21, 3.5))
for ax, (code, g) in zip(axes, df.groupby("dx")):
    ax.imshow(Image.open(paths[g["image_id"].iloc[0]]))
    ax.set_title(code)
    ax.axis("off")
plt.show()

## Smoke test (1 epoch)

Run this first. It should print `Device: cuda`, the split sizes, one epoch line, a classification report and "Done". If it fails, fix that before the long run (the README lists common errors).

In [ ]:
!python train.py --data_dir "{DATA}" --out_dir /content/test_run --epochs 1

## Full training (20 epochs)

This takes roughly 20-40 minutes. Keep this browser tab open. The best checkpoint and all result files are written to Google Drive as training progresses. `train.py` cannot resume, so if the session disconnects, re-run the cells from the top and then this one.

In [ ]:
!python train.py --data_dir "{DATA}" --out_dir "{OUT_DIR}" --epochs 20

## Look at the results

In [ ]:
import json
from IPython.display import Image as IPImage, display

with open(f"{OUT_DIR}/metrics.json") as f:
    m = json.load(f)
print({k: round(v, 3) for k, v in m.items() if isinstance(v, float)})
for name in ("training_curves.png", "confusion_matrix.png"):
    display(IPImage(f"{OUT_DIR}/{name}"))

## Download the results

You can also take the `models` folder straight from Google Drive (`My Drive/dermavision/models`).

In [ ]:
import shutil
from pathlib import Path
from google.colab import files

shutil.make_archive("/content/models", "zip", root_dir=Path(OUT_DIR).parent, base_dir=Path(OUT_DIR).name)
files.download("/content/models.zip")

Unzip `models.zip` into the project folder on your computer so that `models/dermavision_best.pt` sits next to `app.py`, then follow "Run it on your computer" in the README to start the app.